# INS whole-window Haufe pattern visualization

Temporal heatmaps and MNI insula brain maps for merged pseudo-subjects `INSl` / `INSr`.

- **LexicalDelay:** Repeat + Decision; lexicality / phoneme / articulator
- **PhonemeSequence:** Repeat; phoneme / articulator

Heatmaps mask non-significant timepoints to NaN; channels are ordered by NMF `functional_cluster` blocks with cluster-colored y-tick labels.

Brain maps: **one figure per decoding type** — 2×4 (Left=`INSl` / Right=`INSr` × phases) on `cvs_avg35_inMNI152`. Significant: red = sustained, gold = intermediate, blue = sensory; non-significant: gray.

Channels whose significant samples are ≥95% pre-onset (`t < 0`) are dropped from significance (touching pre-onset alone is not enough).

Style: [`docs/PLOTTING_STYLE.md`](../docs/PLOTTING_STYLE.md).

In [9]:
%matplotlib inline

import logging
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import mne
import pandas as pd
import pyvista as pv
from IPython.display import display

# Resolve repo root from this notebook file (cwd-independent).
_NB_DIR = Path.cwd()
for _cand in (_NB_DIR, _NB_DIR.parent, *_NB_DIR.parents):
    if (_cand / 'src' / 'paths.py').is_file():
        ROOT = _cand.resolve()
        break
else:
    ROOT = Path('..').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import importlib

from src.paths import PROJECT_ROOT, RESULTS_ROOT
from src.decoding.prepare_insula_decoding_dataset import PSEUDO_SUBJECTS
import src.decoding.viz_insula_patterns as _viz_mod
importlib.reload(_viz_mod)  # pick up disk edits; avoid stale set_3d_backend("agg")
from src.decoding.viz_insula_patterns import (
    CLUSTER_COLORS,
    CLUSTER_LABELS,
    CLUSTERS,
    FEATURE_COLORS,
    LEXICAL_GRID,
    PHASES,
    PHONEME_GRID,
    load_assignments,
    pattern_h5_path,
    plot_feature_overlay_brain,
    plot_single_feature_brain,
    plot_subject_phase_heatmaps,
    run_all_figures,
)
from src.univariate.viz_mean import BrainSurfaceContext

os.environ.setdefault('PYVISTA_OFF_SCREEN', 'true')
if os.environ.get('MNE_3D_BACKEND', '').lower() == 'agg':
    os.environ['MNE_3D_BACKEND'] = 'notebook'
pv.OFF_SCREEN = True
mne.viz.set_3d_backend('notebook')

logging.basicConfig(level=logging.INFO, format='%(levelname)s %(message)s')
logger = logging.getLogger(__name__)

ROOT = PROJECT_ROOT
from src.paths import img_dir
OUT_DIR = img_dir('insula_patterns')
_src = Path(_viz_mod.__file__).read_text()
print('PROJECT_ROOT:', PROJECT_ROOT)
print('viz module:', _viz_mod.__file__)
print('3d backend:', mne.viz.get_3d_backend())
print('source still has set_3d_backend("agg")?', 'set_3d_backend("agg")' in _src)

PROJECT_ROOT: /hpc/group/coganlab/nanlinshi/insula-functional
viz module: /hpc/group/coganlab/nanlinshi/insula-functional/src/decoding/viz_insula_patterns.py
3d backend: notebook
source still has set_3d_backend("agg")? False


In [10]:
cm = 1 / 2.54
plt.rcParams['svg.fonttype'] = 'none'

fontsize = 7
fontdict = dict(fontsize=fontsize)

# Re-export style constants for inline tweaks / legends
red = CLUSTER_COLORS['sustained_ramping']
gold = CLUSTER_COLORS['intermediate']
blue = CLUSTER_COLORS['sensory_transient']

print('CLUSTERS:', CLUSTERS)
print('Cluster colors:', CLUSTER_COLORS)
print('Feature colors:', FEATURE_COLORS)
print('Output dir:', OUT_DIR)

CLUSTERS: ('sustained_ramping', 'intermediate', 'sensory_transient')
Cluster colors: {'sustained_ramping': '#A9373B', 'intermediate': '#C4A35A', 'sensory_transient': '#2369BD'}
Feature colors: {'lexicality': '#2369BD', 'phoneme': '#CC8963', 'articulator': '#A9373B'}
Output dir: /hpc/group/coganlab/nanlinshi/insula-functional/img/insula_patterns


## Load assignments and inventory pattern H5 files

In [11]:
assignments = load_assignments(PROJECT_ROOT)
display(assignments.head())

rows = []
for grid in (LEXICAL_GRID, PHONEME_GRID):
    for subject in PSEUDO_SUBJECTS:
        for description in grid['descriptions']:
            for feature in grid['features']:
                for phase in PHASES:
                    path = pattern_h5_path(
                        PROJECT_ROOT,
                        task=grid['task'],
                        subject=subject,
                        feature=feature,
                        phase=phase,
                        description=description,
                    )
                    rows.append({
                        'task': grid['task'],
                        'subject': subject,
                        'description': description,
                        'feature': feature ,
                        'phase': phase,
                        'exists': path.exists(),
                        'path': str(path),
                    })

inventory = pd.DataFrame(rows)
print(f"Found {inventory.exists.sum()} / {len(inventory)} pattern H5 files")
display(inventory.groupby(['task', 'feature']).exists.sum())

,channel,functional_cluster,hemi,x,y,z
0,D0022_LMIF3-4,sustained_ramping,L,-36.8260,3.42880,-28.518790
1,D0022_LPIF3-4,intermediate,L,-33.9695,-3.02540,-19.234312
2,D0023_R2IF1-2,intermediate,R,35.7033,7.35276,-0.947400
3,D0027_LAI1-2,sustained_ramping,L,-33.4346,25.11575,-21.905890
4,D0028_LAI1-2,sustained_ramping,L,-32.7839,29.09270,-30.771500


Found 0 / 64 pattern H5 files


task             feature    
LexicalDelay     articulator    0
                 lexicality     0
                 phoneme        0
PhonemeSequence  articulator    0
                 phoneme        0
Name: exists, dtype: int64

## Temporal heatmaps

`pattern` is shown only where `pattern_mask` is true (else NaN). For OvR features (phoneme / articulator), classes are aggregated: any class significant at a channel×time counts as significant, and signed patterns are averaged across significant classes.

Channels with ≥95% of significant samples before onset (`t < 0`) are removed before plotting.

In [12]:
for grid in (LEXICAL_GRID, PHONEME_GRID):
    task = grid['task']
    for description in grid['descriptions']:
        for feature in grid['features']:
            plot_subject_phase_heatmaps(
                PROJECT_ROOT, assignments, OUT_DIR,
                task=task, description=description, feature=feature,
            )

2026-07-28 10:43:05 - WARNING - Skip heatmap (no data): LexicalDelay Repeat lexicality
2026-07-28 10:43:05 - WARNING - Skip heatmap (no data): LexicalDelay Repeat phoneme
2026-07-28 10:43:05 - WARNING - Skip heatmap (no data): LexicalDelay Repeat articulator
2026-07-28 10:43:05 - WARNING - Skip heatmap (no data): LexicalDelay Decision lexicality
2026-07-28 10:43:05 - WARNING - Skip heatmap (no data): LexicalDelay Decision phoneme
2026-07-28 10:43:05 - WARNING - Skip heatmap (no data): LexicalDelay Decision articulator
2026-07-28 10:43:05 - WARNING - Skip heatmap (no data): PhonemeSequence Repeat phoneme
2026-07-28 10:43:05 - WARNING - Skip heatmap (no data): PhonemeSequence Repeat articulator


## Spatial single-feature brain maps

**One figure per decoding type** (`task × description × feature`), not per INSl/INSr.

Layout: **2 × 4** — row 1 Left (`INSl`) / row 2 Right (`INSr`); columns = Stimulus → Delay → Go → Response. Significant: red = sustained / ramping, gold = intermediate, blue = sensory / transient. Non-significant: gray.

In [13]:
ctx = BrainSurfaceContext()

# One figure per task × description × feature (Left=INSl, Right=INSr; cols=phases).
for grid in (LEXICAL_GRID, PHONEME_GRID):
    task = grid['task']
    for description in grid['descriptions']:
        for feature in grid['features']:
            plot_single_feature_brain(
                PROJECT_ROOT, assignments, OUT_DIR,
                task=task, description=description, feature=feature,
                ctx=ctx,
            )

Reading labels from parcellation...
   read 75 labels from /cwork/ns458/ECoG_Recon/cvs_avg35_inMNI152/label/lh.aparc.a2009s.annot
   read 75 labels from /cwork/ns458/ECoG_Recon/cvs_avg35_inMNI152/label/rh.aparc.a2009s.annot
2026-07-28 10:43:05 - WARNING - Skip brain single (missing): LexicalDelay Repeat lexicality
2026-07-28 10:43:05 - WARNING - Skip brain single (missing): LexicalDelay Repeat phoneme
2026-07-28 10:43:05 - WARNING - Skip brain single (missing): LexicalDelay Repeat articulator
2026-07-28 10:43:05 - WARNING - Skip brain single (missing): LexicalDelay Decision lexicality
2026-07-28 10:43:05 - WARNING - Skip brain single (missing): LexicalDelay Decision phoneme
2026-07-28 10:43:05 - WARNING - Skip brain single (missing): LexicalDelay Decision articulator
2026-07-28 10:43:05 - WARNING - Skip brain single (missing): PhonemeSequence Repeat phoneme
2026-07-28 10:43:05 - WARNING - Skip brain single (missing): PhonemeSequence Repeat articulator


## Multi-feature overlap brain maps (deferred)

Not run for now — finish per-feature brains first. Overlay helper stays in `viz_insula_patterns.py` for later.

## Cross-task Repeat union (LexicalDelay ∪ PhonemeSequence)

Same **2×4** layout as single-feature brains (rows = Left/Right; cols = phases). On each panel, significant electrodes are the **union** across LexicalDelay and PhonemeSequence Repeat for that feature — no task-level color/shape; cluster colors unchanged (red=sustained, gold=intermediate, blue=sensory). Per-task `*_brain_single` figures are left as-is.


In [14]:
from src.decoding.viz_insula_patterns import plot_cross_task_repeat_brain

cross_paths = []
for feature in ('phoneme', 'articulator'):
    path = plot_cross_task_repeat_brain(
        PROJECT_ROOT, assignments, OUT_DIR,
        feature=feature, ctx=ctx,
    )
    cross_paths.append((feature, path))
    print(feature, '->', path)

2026-07-28 10:43:05 - WARNING - Skip cross-task brain (missing): LexicalDelay+PhonemeSequence Repeat phoneme
phoneme -> None
2026-07-28 10:43:05 - WARNING - Skip cross-task brain (missing): LexicalDelay+PhonemeSequence Repeat articulator
articulator -> None


In [15]:
# One overlay figure per task×description (Left=INSl, Right=INSr; cols=phases).
for grid in (LEXICAL_GRID, PHONEME_GRID):
    task = grid['task']
    for description in grid['descriptions']:
        path = plot_feature_overlay_brain(
            PROJECT_ROOT, assignments, OUT_DIR,
            task=task, description=description,
            features=tuple(grid['features']), ctx=ctx,
        )
        print(task, description, '->', path)


2026-07-28 10:43:05 - WARNING - Skip brain overlay (missing): LexicalDelay Repeat
LexicalDelay Repeat -> None
2026-07-28 10:43:05 - WARNING - Skip brain overlay (missing): LexicalDelay Decision
LexicalDelay Decision -> None
2026-07-28 10:43:05 - WARNING - Skip brain overlay (missing): PhonemeSequence Repeat
PhonemeSequence Repeat -> None


## Batch driver (optional)

Equivalent one-liner for Slurm / CLI use:

```bash
conda run -n ieeg python -m src.decoding.viz_insula_patterns
```

In [16]:
saved = sorted(OUT_DIR.glob('*.svg'))
print(f'Saved {len(saved)} SVG figures under {OUT_DIR}')
for path in saved[:12]:
    print(' ', path.name)
if len(saved) > 12:
    print(f'  ... and {len(saved) - 12} more')

Saved 21 SVG figures under /hpc/group/coganlab/nanlinshi/insula-functional/img/insula_patterns
  LexicalDelay_Decision_articulator_brain_single.svg
  LexicalDelay_Decision_articulator_union_heatmap.svg
  LexicalDelay_Decision_brain_overlay.svg
  LexicalDelay_Decision_lexicality_binary_heatmap.svg
  LexicalDelay_Decision_lexicality_brain_single.svg
  LexicalDelay_Decision_phoneme_brain_single.svg
  LexicalDelay_Decision_phoneme_union_heatmap.svg
  LexicalDelay_Repeat_articulator_brain_single.svg
  LexicalDelay_Repeat_articulator_union_heatmap.svg
  LexicalDelay_Repeat_brain_overlay.svg
  LexicalDelay_Repeat_lexicality_binary_heatmap.svg
  LexicalDelay_Repeat_lexicality_brain_single.svg
  ... and 9 more
